<a href="https://colab.research.google.com/github/giuliannaac14/modelo-andamio-LCA/blob/main/modelo_andamio_LCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Modelo

In [ ]:
# ==============================================================================
# SIMULADOR BIOMECÁNICO INTERACTIVO 3D: ANDAMIO EN ARANDELA + INJERTO LCA
# Autor: Asistente de Bioingeniería IA - Anatomía Completa (Injerto + Andamio)
# Plataforma: Google Colab (Python 3 / ipywidgets / Plotly 3D)
# ==============================================================================

# --- PASO 1: INSTALACIÓN Y CONFIGURACIÓN DE LIBRERÍAS ---
print("⚙️ [1/5] Instalando librerías interactivas y de modelado 3D...")
!pip install trimesh scikit-image plotly ipywidgets -q

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from skimage.measure import marching_cubes
import trimesh
from google.colab import files
import ipywidgets as widgets
from ipywidgets import interact, Layout
import warnings
warnings.filterwarnings('ignore')

print("✅ Entorno interactivo preparado exitosamente.\n")

# --- PASO 2: BASE DE DATOS DE BIOMATERIALES Y CARGAS DE LA RODILLA ---
print("🦴 [2/5] Cargando bases de datos anatómicas y biomecánicas...")

biomateriales = {
    "SF + Hidroxiapatita (HA) [Refuerzo Óseo]": {"E_0": 2500.0, "Resistencia": 65.0, "Desc": "Matriz mineralizada de alta rigidez ideal para osteointegración en la pared del túnel óseo."},
    "SF + Colágeno Tipo I [Matriz Flexible]":    {"E_0": 150.0,  "Resistencia": 12.0, "Desc": "Matriz biológica hiperelástica que favorece la adhesión de fibroblastos del LCA."},
    "Nanocompuesto Sintético (PLA + Grafeno)": {"E_0": 3200.0, "Resistencia": 80.0, "Desc": "Polímero sintético reforzado de máxima resistencia estructural y lenta degradación."},
    "Tejido LCA Nativo Sano (Referencia)":     {"E_0": 200.0,  "Resistencia": 35.0, "Desc": "Propiedades fisiológicas naturales del ligamento cruzado anterior humano."}
}

cargas_rodilla = {
    "Rehabilitación / Marcha Lenta (300 N)": 300.0,
    "Marcha Normal Fisiológica (600 N)":     600.0,
    "Trote Ligero / Subir Escaleras (1200 N)": 1200.0,
    "Aterrizaje de Salto / Deporte (1800 N)":  1800.0
}

# --- PASO 3: GENERACIÓN GEOMÉTRICA DEL ANDAMIO Y DEL INJERTO TENDINOSO ---
print("📐 [3/5] Generando geometrías 3D (Andamio Giroide + Haz de Injerto LCA)...")
D_ext = 10.0   # Diámetro externo (hueso)
D_int = 7.0    # Diámetro interno (luz para el tendón)
Altura = 8.0   # Longitud del anillo en el túnel
res_viz = 70   # Resolución optimizada para fluidez interactiva

# 1. Geometría del Andamio (Giroide)
x = np.linspace(-D_ext/2, D_ext/2, res_viz)
y = np.linspace(-D_ext/2, D_ext/2, res_viz)
z = np.linspace(0, Altura, int(res_viz * (Altura/D_ext)))
X, Y, Z = np.meshgrid(x, y, z)

R = np.sqrt(X**2 + Y**2)
mascara = (R >= D_int/2) & (R <= D_ext/2)

k = 3.2
giroide = np.sin(k*X)*np.cos(k*Y) + np.sin(k*Y)*np.cos(k*Z) + np.sin(k*Z)*np.cos(k*X)
r_norm = (R - D_int/2) / (D_ext/2 - D_int/2)
umbral = 0.58 * r_norm - 0.28
matriz_solida = (giroide > umbral) & mascara

verts_and, caras_and, _, _ = marching_cubes(matriz_solida, level=0.5, spacing=(x[1]-x[0], y[1]-y[0], z[1]-z[0]))
verts_and[:, 0] -= D_ext/2
verts_and[:, 1] -= D_ext/2
radios_verts = np.sqrt(verts_and[:, 0]**2 + verts_and[:, 1]**2)
dist_norm_verts = (radios_verts - D_int/2) / (D_ext/2 - D_int/2)

# 2. Geometría del Injerto de Tendón (Cilindro anatómico central que atraviesa el implante)
# Radio 95% del diámetro interno para dejar una ligera luz visual y evitar que las mallas colisionen
malla_injerto = trimesh.creation.cylinder(radius=(D_int/2) * 0.95, height=Altura + 5.0, sections=40)
malla_injerto.apply_translation([0, 0, Altura/2.0]) # Centrar verticalmente en el túnel

# --- PASO 4: EXPORTACIÓN DE ARCHIVO STL (SOLO ANDAMIO PARA IMPRESIÓN/FEA) ---
print("📦 [4/5] Exportando archivo STL del andamio poroso para laboratorio...")
malla_export = trimesh.Trimesh(vertices=verts_and, faces=caras_and)
nombre_stl = "Andamio_LCA_Arandela_Biomimetica_Gradiente.stl"
malla_export.export(nombre_stl)
print(f"✅ Archivo guardado como: '{nombre_stl}' (El injerto no se exporta en STL por ser tejido blando biológico).\n")

# --- PASO 5: MOTOR INTERACTIVO BIOMECÁNICO EN TIEMPO REAL ---
print("🎮 [5/5] CARGANDO PANEL DE CONTROL BIOMECÁNICO INTERACTIVO:")
print("="*80)

def simulador_biomecanico_interactivo(material_sel, actividad_sel, tipo_visualizacion):
    props = biomateriales[material_sel]
    fuerza_N = cargas_rodilla[actividad_sel]

    # Cálculos mecánicos locales en el andamio
    porosidad_verts = 0.40 + 0.35 * (dist_norm_verts ** 1.2)
    fraccion_solida_verts = 1.0 - porosidad_verts
    E_eff_verts = props["E_0"] * 0.85 * (fraccion_solida_verts ** 2.0)

    area_contacto = np.pi * D_int * Altura
    tension_base_MPa = fuerza_N / area_contacto
    tension_local_verts = tension_base_MPa * (1.6 - 0.7 * dist_norm_verts)
    factor_seguridad_verts = props["Resistencia"] / tension_local_verts

    # Configuración de mapa de color
    if tipo_visualizacion == "Tensión de Cizallamiento (MPa)":
        valores_color = tension_local_verts
        escala_color = "Inferno"
        titulo_barra = "Tensión<br>(MPa)"
        rango_color = [np.min(tension_local_verts), np.max(tension_local_verts)]
    elif tipo_visualizacion == "Gradiente de Rigidez E_eff (MPa)":
        valores_color = E_eff_verts
        escala_color = "Viridis"
        titulo_barra = "Rigidez<br>E_eff (MPa)"
        rango_color = [np.min(E_eff_verts), np.max(E_eff_verts)]
    else:
        valores_color = factor_seguridad_verts
        escala_color = "RdYlGn"
        titulo_barra = "Factor de<br>Seguridad"
        rango_color = [0.5, 3.0]

    # Diagnóstico Clínico Automatizado
    fs_minimo = np.min(factor_seguridad_verts)
    tension_maxima = np.max(tension_local_verts)

    if fs_minimo < 1.0:
        estado_alerta = f"🚨 RIESGO DE FALLA INMINENTE (Factor de Seguridad mínimo: {fs_minimo:.2f})."
        explicacion_clinica = f"Bajo una carga de {fuerza_N} N ({actividad_sel.split('(')[0].strip()}), la tensión cortante máxima ({tension_maxima:.2f} MPa) en la interfaz del tendón SUPERA la resistencia del material ({props['Resistencia']} MPa). Se produciría el desgarro o desprendimiento del implante."
        color_caja = "#ffe6e6"
        color_texto = "#990000"
    elif fs_minimo < 1.5:
        estado_alerta = f"⚠️ PRECAUCIÓN BIOMECÁNICA (Factor de Seguridad mínimo: {fs_minimo:.2f})."
        explicacion_clinica = f"El implante resiste la carga ({tension_maxima:.2f} MPa vs Resistencia {props['Resistencia']} MPa), pero el margen de seguridad es muy estrecho para soportar ciclos de fatiga repetitivos en la rodilla."
        color_caja = "#fff0b3"
        color_texto = "#806000"
    else:
        estado_alerta = f"🛡️ ESTABLE Y SEGURO (Factor de Seguridad mínimo: {fs_minimo:.2f})."
        explicacion_clinica = f"El material absorbe la carga anatómica con éxito ({tension_maxima:.2f} MPa vs Resistencia {props['Resistencia']} MPa). La fijación entre el haz tendinoso central y el hueso exterior es altamente viable y segura."
        color_caja = "#e6ffe6"
        color_texto = "#006600"

    display(widgets.HTML(f"""
    <div style="background-color: {color_caja}; border-left: 6px solid {color_texto}; padding: 12px; border-radius: 4px; font-family: sans-serif; margin-bottom: 10px;">
        <h4 style="margin: 0 0 5px 0; color: {color_texto}; font-size: 15px;">{estado_alerta}</h4>
        <p style="margin: 0; font-size: 13px; color: #333;"><b>Biomaterial evaluado:</b> {material_sel.split('[')[0]} <br>
        <b>Análisis Biomecánico:</b> {explicacion_clinica}</p>
    </div>
    """))

    # 1. Traza del Andamio Biomimético (Malla porosa externa)
    traza_andamio = go.Mesh3d(
        x=verts_and[:, 0], y=verts_and[:, 1], z=verts_and[:, 2],
        i=caras_and[:, 0], j=caras_and[:, 1], k=caras_and[:, 2],
        colorscale=escala_color,
        intensity=valores_color,
        cmin=rango_color[0], cmax=rango_color[1],
        colorbar_title=titulo_barra,
        name="Andamio Poroso (Giroide)",
        showscale=True,
        opacity=0.95
    )

    # 2. Traza del Injerto de Tendón LCA (Cilindro biológico central)
    traza_injerto = go.Mesh3d(
        x=malla_injerto.vertices[:, 0],
        y=malla_injerto.vertices[:, 1],
        z=malla_injerto.vertices[:, 2],
        i=malla_injerto.faces[:, 0],
        j=malla_injerto.faces[:, 1],
        k=malla_injerto.faces[:, 2],
        color='#e06666',  # Tono biológico rosado/rojizo característico del tendón
        name="Injerto de Tendón (LCA)",
        showlegend=True,
        opacity=0.85
    )

    # Ensamble de la escena 3D combinada
    fig_3d = go.Figure(data=[traza_andamio, traza_injerto])

    fig_3d.update_layout(
        title=dict(text=f"<b>MODELO 3D: INJERTO DE TENDÓN + ANDAMIO BIOMIMÉTICO</b><br><i>Carga anatómica simulada: {actividad_sel} | Material: {material_sel.split('[')[0]}</i>", font=dict(size=13)),
        scene=dict(
            xaxis_title='Eje X (mm)', yaxis_title='Eje Y (mm)', zaxis_title='Longitud Túnel Z (mm)',
            aspectmode='data',
            camera=dict(eye=dict(x=1.6, y=1.5, z=1.2))
        ),
        margin=dict(l=0, r=0, b=0, t=50),
        height=540,
        legend=dict(x=0.02, y=0.95, bgcolor="rgba(255,255,255,0.8)", bordercolor="gray", borderwidth=1)
    )
    fig_3d.show()

menu_material = widgets.Dropdown(options=list(biomateriales.keys()), value="SF + Colágeno Tipo I [Matriz Flexible]", description='<b>Biomaterial:</b>', style={'description_width': 'initial'}, layout=Layout(width='48%'))
menu_actividad = widgets.Dropdown(options=list(cargas_rodilla.keys()), value="Aterrizaje de Salto / Deporte (1800 N)", description='<b>Actividad:</b>', style={'description_width': 'initial'}, layout=Layout(width='48%'))
menu_vista = widgets.Dropdown(options=["Factor de Seguridad (Resistencia / Tensión)", "Tensión de Cizallamiento (MPa)", "Gradiente de Rigidez E_eff (MPa)"], value="Factor de Seguridad (Resistencia / Tensión)", description='<b>Mapa 3D:</b>', style={'description_width': 'initial'}, layout=Layout(width='60%'))

print("👇 SELECCIONA EL MATERIAL, LA CARGA DE LA RODILLA Y LA VISTA 3D EN LOS MENÚS DE ABAJO:")
interact(simulador_biomecanico_interactivo, material_sel=menu_material, actividad_sel=menu_actividad, tipo_visualizacion=menu_vista);

# --- DESCARGA AUTOMÁTICA DEL ARCHIVO STL ---
print("\n📥 Descargando archivo STL del andamio generado...")
try:
    files.download(nombre_stl)
except Exception as e:
    print(f"⚠️ El archivo '{nombre_stl}' está listo en tu panel de carpetas a la izquierda en Colab.")

⚙️ [1/5] Instalando librerías interactivas y de modelado 3D...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 117.0 MB/s eta 0:00:00
✅ Entorno interactivo preparado exitosamente.

🦴 [2/5] Cargando bases de datos anatómicas y biomecánicas...
📐 [3/5] Generando geometrías 3D (Andamio Giroide + Haz de Injerto LCA)...
📦 [4/5] Exportando archivo STL del andamio poroso para laboratorio...
✅ Archivo guardado como: 'Andamio_LCA_Arandela_Biomimetica_Gradiente.stl' (El injerto no se exporta en STL por ser tejido blando biológico).

🎮 [5/5] CARGANDO PANEL DE CONTROL BIOMECÁNICO INTERACTIVO:
👇 SELECCIONA EL MATERIAL, LA CARGA DE LA RODILLA Y LA VISTA 3D EN LOS MENÚS DE ABAJO:


interactive(children=(Dropdown(description='<b>Biomaterial:</b>', index=1, layout=Layout(width='48%'), options…


📥 Descargando archivo STL del andamio generado...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>